In [ ]:
### Kenyan Parliamentary Hansard Analysis Script

## Nicolas de Zamaroczy

## The purpose of this Python script is to perform simple quantitative text analysis of the parsed Kenyan parliamentary hansards.
# In particular, we track sentiment analysis in a variety of ways:
# - over time;
# - by speaker; and
# - by political party.

# The current goal is to see if parliamentary speech is a leading or lagging indicator of political violence in Kenya.

In [ ]:
### PRELIMINARIES

# --- Dependency Checks ---
# This block checks if several required libraries are installed.
# If not, it uses Python's package manager (pip) to install them.
# This makes the script more portable and easier for others to run.
try:
    import numpy as np # NumPy, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("NumPy not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy"])
    import numpy as np
    print("Installation complete.")

try:
    import pandas as pd # Pandas, a core package used to structure the data
except ImportError:
    import subprocess
    import sys
    print("Pandas not found. Installing...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd
    print("Installation complete.")

try:
    from pathlib import Path # PathLib, used to read/write files to a given folder
except ImportError:
    import subprocess
    import sys
    print("PathLib not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pathlib[Path]"])
    from pathlib import Path
    print("Installation complete.")

try:
    from tqdm import tqdm # tqdm, used to create a progress bar
except ImportError:
    import subprocess
    import sys
    print("tqdm not found. Installing...")
    # It's good to install the optional dependency for faster performance.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm[tqdm]"])
    from tqdm import tqdm
    print("Installation complete.")

# Enable tqdm for Pandas
tqdm.pandas()

# Disable Future Deprecation Warnings from getting logged
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

# On Windows, Anaconda's Python ships an old Microsoft C++ runtime (msvcp140.dll) that
# recent PyTorch builds can't use ("DLL initialization routine failed" for c10.dll).
# Loading the newer copy from Windows first, before PyTorch is imported, avoids this.
import os
if os.name == "nt":
    import ctypes
    _msvcp = os.path.join(os.environ.get("SystemRoot", r"C:\Windows"), "System32", "msvcp140.dll")
    if os.path.exists(_msvcp):
        ctypes.WinDLL(_msvcp)

In [ ]:
# Load the parsed speeches from the pipeline's database (built by `hansard parse`)

import json
import sqlite3
from contextlib import closing

# --- Settings ---
CHAMBER = "senate"  # "senate", "national_assembly" or "joint"
# Number of speeches to analyse, picked at random. None analyses every speech, which
# takes many hours on a CPU; a few thousand is enough for a quick test run.
SAMPLE_SIZE = None
# The multilingual zero-shot classification cell near the end takes over 24 hours on
# a CPU, so it only runs when this is True.
RUN_MULTILINGUAL_ZERO_SHOT = False

# Current notebook directory
notebook_dir = Path.cwd()
# Path to the database
db_path = notebook_dir.parent / "outputs" / "hansard.db"
# Check existence and raise an error if missing
if not db_path.exists():
    raise FileNotFoundError(f"Database not found: {db_path}. Run `hansard parse` first.")

# One row per speech, with its document and (when matched) member details joined in
query = """
SELECT s.id AS speech_id, d.filename, d.date, d.chamber, d.sitting,
       s.speaker_raw AS speaker, m.full_name AS matched_name, m.party, m.county,
       m.constituency, s.language_distribution, s.primary_language, s.text AS content
FROM speeches s
JOIN documents d ON d.id = s.document_id
LEFT JOIN members m ON m.id = s.member_id
WHERE d.doc_type = 'hansard' AND d.chamber = ?
ORDER BY d.date, d.id, s.seq
"""
with closing(sqlite3.connect(db_path)) as conn:
    df_blocks = pd.read_sql_query(query, conn, params=[CHAMBER])

if SAMPLE_SIZE:
    df_blocks = (df_blocks.sample(n=min(SAMPLE_SIZE, len(df_blocks)), random_state=42)
                 .sort_values(["date", "speech_id"], ignore_index=True))

# Paragraphs are separated by newlines; the models work on plain running text
df_blocks["content"] = df_blocks["content"].str.replace(r"[\r\n]+", " ", regex=True)

# Language mix as a dict, e.g. {"en": 0.8, "sw": 0.2}
df_blocks["language_distribution"] = df_blocks["language_distribution"].map(json.loads)

# Ensure date column is a datetime
df_blocks["date"] = pd.to_datetime(df_blocks["date"])

df_blocks

In [ ]:
# Split long speeches into chunks for the language models

# The sentiment and zero-shot models read at most 512 tokens, so longer speeches are
# split into chunks of up to 480 words (roughly 512 tokens). Every chunk keeps its
# speech's details, and `chunk` numbers the chunks within a speech from 0.

# (Party, county and constituency now come from the database's members table, so the
# Google Sheet no longer needs to be merged in here.)

MAX_WORDS = 480

def chunk_words(text, max_words=MAX_WORDS):
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)] or [""]

df_blocks["content"] = df_blocks["content"].map(chunk_words)
df_blocks = df_blocks.explode("content", ignore_index=True)
df_blocks["chunk"] = df_blocks.groupby("speech_id").cumcount()

df_blocks

In [ ]:
# Perform multilingual sentiment analysis using batching

# Here is a direct link to tabularisai's multilingual-sentiment-analysis package, hosted on HuggingFace:
# https://huggingface.co/tabularisai/multilingual-sentiment-analysis

# Tabularisai's MSA tool automatically detects speech, so in theory it doesn't need the information contained in language_distribution.

# Import pipeline
from transformers import pipeline

# Set up the pipeline
pipe = pipeline(
    "text-classification",
    model="tabularisai/multilingual-sentiment-analysis"
)

# Set up batching
texts = df_blocks["content"].astype(str).tolist()
batch_size = 32

# Run pipeline
all_results = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    #results = pipe(batch)
    results = pipe(batch, truncation=True, max_length=510) # This has a saftey valve built in to not hit the 512 max tokens limit Tabularisai has, but shouldn't be necessary since the chunking cell splits speeches into 480-word chunks
    all_results.extend(results)

df_blocks["sentiment"] = [r["label"] for r in all_results]
df_blocks["confidence_score"] = [r["score"] for r in all_results]

df_blocks

In [ ]:
# Visualizing the distribution of confidence scores

palette = {
    "Very Negative": "#d73027",
    "Negative": "#fc8d59",
    "Neutral": "#999999",
    "Positive": "#91cf60",
    "Very Positive": "#1a9850"
}

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.histplot(
    data=df_blocks,
    x="confidence_score",
    hue="sentiment",
    bins=30,
    kde=True,
    palette=palette
)
plt.title("Distribution of Confidence Scores by Class")
plt.xlabel("Confidence Score")
plt.ylabel("Count")
plt.show()

In [ ]:
# Checking to see if sentiment analysis passes the sniff test

# Define all sentiment classes
sentiment_classes = ["Very Negative", "Negative", "Neutral", "Positive", "Very Positive"]

# Function to safely sample up to 5 rows
def safe_sample(df, label, n=5):
    subset = df[df["sentiment"] == label]
    return subset.sample(n=min(len(subset), n), random_state=42)

# Change display settings to allow full text to show instead of truncating
pd.set_option('display.max_colwidth', None)

# Print sample content for each class
for label in sentiment_classes:
    print(f"\n--- {label} ---")
    sample = safe_sample(df_blocks, label, 5)
    display(sample[["content", "confidence_score"]])  # use display() in Jupyter


In [ ]:
# Construct a single measure of sentiment for over-time analysis

# Reset our display options to their defaults
pd.reset_option("display.max_rows")
pd.reset_option("display.max_colwidth")

scale_map = {
    "Very Negative": -1,
    "Negative": -0.5,
    "Neutral": 0,
    "Positive": 0.5,
    "Very Positive": 1
}

df_blocks["sentiment_scaled"] = df_blocks["sentiment"].map(scale_map) * df_blocks["confidence_score"]

# Exclude all the neutral sentiment blocks (which is the overwhelming majority, so this greatly pares down df_blocks)
df_blocks_w_sentiment = df_blocks[df_blocks['sentiment']!='Neutral'].copy()

df_blocks_w_sentiment

In [ ]:
# Group by week
df_blocks_w_sentiment["date"] = pd.to_datetime(df_blocks_w_sentiment["date"])

# Group by week (ending on Sunday by default) and average the scaled sentiment
weekly_sentiment = (
    df_blocks_w_sentiment
    .set_index("date")
    .resample("W")["sentiment_scaled"]
    .mean()
    .reset_index()
)

weekly_sentiment

In [ ]:
# Visualize

import plotly.express as px

fig = px.line(
    weekly_sentiment,
    x="date",
    y="sentiment_scaled",
    title="Weekly Sentiment Over Time",
    labels={"date": "Week", "sentiment_scaled": "Avg Sentiment (scaled)"}
)

fig.update_traces(mode="lines+markers")  # adds points to each week
fig.show()

In [ ]:
# ==============================================================================
# --- VALIDATION QUERIES ---
# Useful queries are stored here for subsequent use.
# ==============================================================================

# Find examples of code-switching (i.e. when a speaker switches languages mid-speech)
#df_codeswitching = df_blocks[df_blocks['language_distribution'].map(len) > 1]

# Check that there isn't a language bias in our sentiment analysis

# Look to see which parties have the highest and lowest sentiment

#  Is it worth introducing confidence thresholds


In [ ]:
# ==============================================================================
# --- ONE SHOT ENCODING ---
# ==============================================================================

############### This is the right approach for multilingual zero-shot classification, but takes way too long to run locally >24h).

# We use the ############### Hugging Face transformer to do zero-shot classification since it is multilingual and supports Swahili as a language.
# See ########### for documentation

if RUN_MULTILINGUAL_ZERO_SHOT:
    # Set up the pipeline
    from transformers import pipeline

    classifier = pipeline("zero-shot-classification"
                          ,model="MoritzLaurer/multilingual-MiniLMv2-L12-mnli-xnli")

    # Define topic labels for zero-shot classification
    labels = [
        "parliamentary process",
        "economy and finance",
        "education",
        "healthcare",
        "foreign affairs",
        "security and defense",
        "governance and corruption",
        "environment and climate",
        "human rights and equality",
        "infrastructure projects",
        "agriculture and food security",
        "devolution and local government"
    ]

    # Set up batching
    batch_size = 8
    results = []

    # Execute
    for i in tqdm(range(0, len(df_blocks), batch_size)):
        batch = df_blocks["content"].iloc[i:i+batch_size].tolist()
        batch_results = classifier(batch, labels)  # supports batch inputs

        for res in batch_results:
            results.append((res["labels"][0], res["scores"][0]))

    df_blocks[["predicted_label", "confidence"]] = pd.DataFrame(results, index=df_blocks.index)
else:
    print("Skipped: set RUN_MULTILINGUAL_ZERO_SHOT = True in the settings to run this "
          "(over 24 hours on a CPU).")

In [ ]:
## English-only speedy zero-shot classification

# English-only subset
df_en = df_blocks[df_blocks["language_distribution"].map(lambda d: d == {"en": 1.0})].copy()

# Truncate to keep only first 500 characters
df_en["content_trunc"] = df_en["content"].str.slice(0, 500)

# Import pipeline
from transformers import pipeline

# Load a fast, English-only zero-shot model
classifier = pipeline(
    "zero-shot-classification",
    model="valhalla/distilbart-mnli-12-3",
    device=-1
)

# # Debugging block
# test_text = "The government proposed new tax reforms to boost the economy."
# labels = [
#     "economy and finance",
#     "education",
#     "security and defense"
# ]
# result = classifier(test_text, labels)
# print(result)

# Define labels
labels = [
    "parliamentary process",
    "economy and finance",
    "education",
    "healthcare",
    "foreign affairs",
    "security and defense",
    "governance and corruption",
    "environment and climate",
    "human rights and equality",
    "infrastructure projects",
    "agriculture and food security",
    "devolution and local government"
]

# Batched zero-shot classification
batch_size = 8
predicted_labels = []
confidences = []

for i in tqdm(range(0, len(df_en), batch_size)):
    batch_texts = df_en["content_trunc"].iloc[i:i+batch_size].tolist()
    batch_results = classifier(batch_texts, candidate_labels=labels)
    
    # Ensure consistent output format
    if isinstance(batch_results, dict):  # Single input → single dict
        batch_results = [batch_results]
    
    for r in batch_results:
        predicted_labels.append(r["labels"][0])
        confidences.append(r["scores"][0])

# Assign back
df_en = df_en.assign(predicted_label=predicted_labels, confidence=confidences)